# Unit 10 / Chapter 10: AI for Quantum Computing

> **Main Learning Objective:** Flip the usual question and study how classical AI is used to *build* better quantum computers. See how ML helps in circuit design, calibration, error mitigation, and control.

| Section | Topic | Why it matters |
|---|---|---|
| 10.1 | The reverse question | AI helping quantum, not the other way around |
| 10.2 | ML for quantum circuit synthesis | Auto-design of shallow circuits for a target unitary |
| 10.3 | ML for quantum calibration and control | Neural networks that tune pulses on real hardware |
| 10.4 | ML for error mitigation | Learning noise models and undoing them |

By the end of this unit you should be able to:

1. Explain why AI for Quantum is a real and active research field.
2. Name at least two places in a quantum computing stack where machine learning is used today.
3. Describe circuit synthesis at a high level.
4. Understand the difference between error correction (Unit 7) and error mitigation.
5. Identify one or two open problems where more AI would be genuinely useful for quantum.

---

## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100
print("Imports done.")

---
## Course check-in

This logs that you started **Unit 10**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 10
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 10.1: The Reverse Question

Every earlier unit asked: how does quantum help AI. Unit 10 asks the reverse: how does AI help quantum?

Turns out: a lot. Classical machine learning is used inside modern quantum labs and cloud offerings in at least these places:

| Where in the stack | What ML does |
|---|---|
| Circuit compilation | Auto-simplify user circuits into hardware-native gates |
| Circuit synthesis | Given a target unitary, find the shortest circuit that implements it |
| Pulse-level control | Tune microwave pulses to reduce gate error |
| Calibration | Learn drift models and re-tune qubits over time |
| Readout classification | ML classifier turns analog signals into 0 or 1 |
| Error mitigation | Post-process noisy outputs to estimate what the noise-free output would be |
| Noise characterization | Learn a noise model of the device from experiments |
| Scheduling | Optimize how to schedule jobs on a quantum machine |

This is not a research curiosity. IBM, Google, Quantinuum, IonQ, PsiQuantum, and every other major quantum vendor uses ML somewhere in their stack today.

---
# Section 10.2: Machine Learning for Circuit Synthesis

Given a target unitary U (say, the ideal action you want on 4 qubits), find a short sequence of native gates whose product equals U to some accuracy. This is called **circuit synthesis** and is critical because shallow circuits are more reliable.

Modern approaches include:

- Reinforcement learning agents whose actions are gate insertions.
- Neural network guided search over gate sequences.
- Transformer-based sequence-to-sequence models.

Below we set up a tiny toy version: given a random single qubit target unitary, find a sequence of Ry and Rz rotations that approximates it.

In [ ]:
# Given a random target 1-qubit unitary U_target, fit U_theta = Rz(a) Ry(b) Rz(c)
# by minimizing || U_theta - U_target ||_F.
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)

# Random target: pick a random unitary via QR decomposition
np.random.seed(3)
A = np.random.randn(2, 2) + 1j*np.random.randn(2, 2)
Q, _ = np.linalg.qr(A)
U_target = Q

def circuit(theta):
    return Rz(theta[0]) @ Ry(theta[1]) @ Rz(theta[2])

def loss(theta):
    return np.linalg.norm(circuit(theta) - U_target)**2

theta = np.array([0.1, 0.1, 0.1])
lr = 0.1
history = []
for step in range(300):
    l = loss(theta); history.append(l)
    g = np.zeros_like(theta)
    for i in range(3):
        t1 = theta.copy(); t1[i] += 1e-3
        t2 = theta.copy(); t2[i] -= 1e-3
        g[i] = (loss(t1) - loss(t2)) / 2e-3
    theta = theta - lr * g

plt.plot(history, color='#5B2C91', lw=2)
plt.xlabel('step'); plt.ylabel('|| U_theta - U_target ||^2')
plt.title('Synthesizing a random 1-qubit unitary with Rz Ry Rz')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"Final error: {history[-1]:.4e}")
print(f"Angles found: a={theta[0]:.3f}, b={theta[1]:.3f}, c={theta[2]:.3f}")

---
# Section 10.3: Machine Learning for Calibration and Control

Every day a quantum computer drifts. Qubit frequencies wander, gates go out of tune, temperature fluctuates. Every major vendor has an automatic recalibration system that runs experiments and adjusts pulse shapes.

Increasingly, that recalibration system uses ML:

- Reinforcement learning to choose which qubit to recalibrate next.
- Bayesian optimization to find optimal pulse parameters.
- Neural networks to predict how the device will drift so recalibration is proactive.

Below we simulate a tiny version: a "qubit" has an unknown optimal pulse duration, and we use gradient descent (a stand-in for ML calibration) to find it.

In [ ]:
# Simulate: a pulse of duration t gives fidelity f(t) = sin^2(t - t_optimal + noise)
# We want to find t_optimal by trying pulses and observing fidelities.

t_optimal_true = 1.4
noise_level = 0.03

def measured_fidelity(t):
    ideal = np.cos((t - t_optimal_true) * np.pi / 2)**2
    return ideal + noise_level * np.random.randn()

t = 0.0
lr = 0.4
history = {'t': [t], 'f': [measured_fidelity(t)]}
for step in range(30):
    # Estimate gradient via one-sided difference (typical in real calibration)
    eps = 0.05
    f_here = measured_fidelity(t)
    f_next = measured_fidelity(t + eps)
    grad = (f_next - f_here) / eps
    t = t + lr * grad
    history['t'].append(t); history['f'].append(measured_fidelity(t))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history['t'], marker='o', color='#5B2C91', lw=2)
axes[0].axhline(t_optimal_true, color='gray', ls='--', label=f'true optimal = {t_optimal_true}')
axes[0].set_title('Calibration parameter over time'); axes[0].set_xlabel('step'); axes[0].set_ylabel('t'); axes[0].legend()
axes[1].plot(history['f'], marker='o', color='#2A9D8F', lw=2)
axes[1].set_title('Measured fidelity over time'); axes[1].set_xlabel('step'); axes[1].set_ylabel('fidelity')
plt.tight_layout(); plt.show()
print(f"Converged near t = {history['t'][-1]:.3f}, true was {t_optimal_true}")

---
# Section 10.4: Machine Learning for Error Mitigation

Error *correction* (Unit 7) protects a quantum computation from errors while it happens. Error *mitigation* runs a noisy computation and then uses classical post processing to guess what the noise-free result would have been.

Common mitigation techniques include:

- **Zero noise extrapolation:** run the circuit at multiple noise levels and extrapolate to zero noise.
- **Probabilistic error cancellation:** learn a noise model, then compute an unbiased estimator that cancels its effect.
- **Learning based mitigation:** train a neural network on (noisy, ideal) pairs and use it to correct new noisy outputs.

All of these use classical ML in an essential way. Below we simulate a simple learning-based mitigator that maps noisy expectation values to ideal ones.

In [ ]:
# Simulate: run 200 random circuits, collect (noisy, ideal) expectation values,
# fit a simple linear map ideal ~ a * noisy + b as our "mitigator."
np.random.seed(4)
N = 200

# 'ideal' expectation values uniform in [-1, 1]
ideals = np.random.uniform(-1, 1, size=N)
# 'noisy' = ideal shrunk toward 0 (typical NISQ noise) + gaussian noise
noisy  = 0.6 * ideals + 0.1 * np.random.randn(N)

# Fit linear mitigator
A = np.vstack([noisy, np.ones(N)]).T
(a, b), _, _, _ = np.linalg.lstsq(A, ideals, rcond=None)

# Apply
mitigated = a * noisy + b

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(noisy, ideals, alpha=0.4, color='#5B2C91', label='noisy raw')
axes[0].plot([-1,1],[-1,1], color='gray', ls='--', label='ideal line')
axes[0].set_xlabel('noisy expectation'); axes[0].set_ylabel('true ideal')
axes[0].set_title('Before mitigation'); axes[0].legend()

axes[1].scatter(mitigated, ideals, alpha=0.4, color='#2A9D8F', label='mitigated')
axes[1].plot([-1,1],[-1,1], color='gray', ls='--')
axes[1].set_xlabel('mitigated'); axes[1].set_title('After learning-based mitigation')
axes[1].set_ylabel('true ideal'); axes[1].legend()
plt.tight_layout(); plt.show()

err_before = np.mean((noisy    - ideals)**2)
err_after  = np.mean((mitigated - ideals)**2)
print(f"MSE before mitigation: {err_before:.4f}")
print(f"MSE after  mitigation: {err_after:.4f}")
print(f"Improvement: {(1 - err_after/err_before)*100:.1f}%")

---
# End-of-Unit Quiz

Ten multiple choice questions. Reveal the answer to check yourself.

**1. AI for Quantum is different from Quantum for AI because it asks:**

A) How can quantum computers train better neural networks
B) How can classical ML improve quantum hardware and software
C) How can we replace all quantum computers with AI
D) How can we prove P = NP

<details><summary>Answer 1</summary>**B).** The direction is reversed: classical ML is used to build and operate quantum computers.</details>

---

**2. Circuit synthesis is the task of:**

A) Running a fixed circuit on hardware
B) Finding a short circuit that implements a given target unitary
C) Correcting quantum errors
D) Measuring a quantum state

<details><summary>Answer 2</summary>**B).** Shorter circuits are more reliable, so the search matters.</details>

---

**3. Which of these is a modern approach to quantum circuit synthesis?**

A) Reinforcement learning
B) Neural network guided search
C) Transformer sequence models
D) All of the above

<details><summary>Answer 3</summary>**D) All of the above.** Circuit synthesis is a hot ML research area.</details>

---

**4. Error mitigation differs from error correction because:**

A) Mitigation prevents errors before they happen
B) Mitigation runs a noisy computation and uses post processing to estimate the noise-free result
C) Mitigation requires a fault tolerant quantum computer
D) Mitigation and correction are the same thing

<details><summary>Answer 4</summary>**B).** Mitigation is classical post processing; correction is real time protection during the computation.</details>

---

**5. Which of these is NOT an ML application inside a real quantum computing stack today?**

A) Readout classification
B) Pulse level control
C) Automatic recalibration
D) Solving the halting problem

<details><summary>Answer 5</summary>**D).** The other three are used in production by real quantum vendors.</details>

---

**6. Why does circuit depth matter so much on NISQ hardware?**

A) Longer circuits look better in code
B) Longer circuits compound gate errors and become unreliable
C) Longer circuits use less power
D) Depth has no effect on reliability

<details><summary>Answer 6</summary>**B).** Each gate has some error probability, so success decays roughly exponentially with depth.</details>

---

**7. Readout classification in a quantum computing stack does what?**

A) Corrects logical errors
B) Turns analog measurement signals into 0 or 1 outcomes
C) Generates random numbers
D) Predicts the next gate to apply

<details><summary>Answer 7</summary>**B).** A learned classifier maps the noisy analog readout waveform to a clean bit outcome.</details>

---

**8. Zero noise extrapolation works by:**

A) Buying better hardware
B) Running the same circuit at several deliberately amplified noise levels and extrapolating to zero noise
C) Ignoring the errors and hoping for the best
D) Increasing the number of qubits

<details><summary>Answer 8</summary>**B).** ZNE is a classical post-processing trick that extrapolates the expectation value to the imagined noise-free limit.</details>

---

**9. Which of these is a common ML tool for tuning quantum pulses?**

A) Bayesian optimization
B) Sorting algorithms
C) Graph coloring
D) LZ compression

<details><summary>Answer 9</summary>**A) Bayesian optimization.** Also common: reinforcement learning and Gaussian process regression.</details>

---

**10. Why is AI for Quantum a genuine research field rather than a marketing phrase?**

A) Because AI is fashionable this decade
B) Because operating a real quantum computer has many concrete ML-shaped bottlenecks
C) Because quantum cannot work without AI
D) Because the term sounds impressive

<details><summary>Answer 10</summary>**B).** Circuit compilation, calibration, drift prediction, readout, and mitigation are all real problems where ML now beats hand-crafted heuristics.</details>

---

## Unit 10 Summary

| Concept | Key idea |
|---|---|
| AI for Quantum | Using classical ML to build and operate quantum computers. |
| Circuit synthesis | Find a short circuit for a target unitary. |
| Circuit compilation | Rewrite a user circuit into hardware native gates. |
| Calibration and control | ML tunes pulses and schedules recalibration. |
| Readout classification | ML classifier decides 0 vs 1 from analog signals. |
| Error mitigation | Post process noisy runs to estimate the noise free answer. |
| Noise characterization | Learn a noise model from experiments. |
| Open questions | Better synthesis, real time calibration at scale, closer to FT via mitigation. |

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 10!")